# Impacto da Retificacao de Perspectiva no OCR de Placas

**Objetivo**: comparar o desempenho do PARSeq Tiny treinado com e sem retificacao
de perspectiva, usando o Cenario A do LRLPR-26 (5 000 placas brasileiras + 5 000 Mercosul).

| Experimento | Treino | Validacao | Teste |
|---|---|---|---|
| **COM retificacao** | crops retificados (corners fornecidos) | crops retificados | crops originais (sem corners) |
| **SEM retificacao** | crops originais | crops originais | crops originais (sem corners) |

O conjunto de teste (3 000 tracks) nao possui anotacao de cantos, entao ambos os
modelos sao avaliados nas imagens cruas — o cenario real de uso.

Hiperparametros iguais aos PARSeqs individuais: 10 epocas, paciencia 3, lr 1e-5, batch 64.
A avaliacao e feita por track: as predicoes dos 5 quadros sao combinadas por MVCP,
como nos notebooks individuais.

## 1. Dependencias

In [ ]:
%pip install -q "timm>=0.6.13"
%pip install -q git+https://github.com/baudm/parseq.git
%pip install -q pytorch-lightning

import torch
import pytorch_lightning as pl

print(f"PyTorch: {torch.__version__}")
print(f"GPU disponivel: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Montar Drive e extrair dataset

In [ ]:
import os
import json
import zipfile
from google.colab import drive

drive.mount('/content/drive')

# ── Ajuste estes caminhos conforme sua configuracao ──
ZIP_PATH    = '/content/drive/MyDrive/<CAMINHO>/LRLPR-26-5opEvJTW.zip'
EXTRACT_DIR = '/content/dataset'

if not os.path.exists(os.path.join(EXTRACT_DIR, 'LRLPR-26-5opEvJTW')):
    print("Extraindo dataset...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(EXTRACT_DIR)
    print("Extracao concluida!")
else:
    print("Dataset ja extraido.")

DATASET_BASE = os.path.join(EXTRACT_DIR, 'LRLPR-26-5opEvJTW')
SCENARIO_A_DIR = os.path.join(DATASET_BASE, 'train', 'Scenario-A')
TEST_DIR = os.path.join(DATASET_BASE, 'test')

print(f"Base: {DATASET_BASE}")
print(f"Cenario A: {SCENARIO_A_DIR}")
print(f"Teste: {TEST_DIR}")

## 3. Explorar dataset e dividir train/val

In [ ]:
import random

random.seed(42)

# Coletar todos os tracks do Cenario A
# Cada entrada: (track_dir, annotations_dict, layout)
all_tracks = {'Brazilian': [], 'Mercosur': []}

for layout in ['Brazilian', 'Mercosur']:
    layout_dir = os.path.join(SCENARIO_A_DIR, layout)
    for track_name in sorted(os.listdir(layout_dir)):
        track_dir = os.path.join(layout_dir, track_name)
        if not os.path.isdir(track_dir):
            continue
        ann_path = os.path.join(track_dir, 'annotations.json')
        with open(ann_path) as f:
            ann = json.load(f)
        all_tracks[layout].append((track_dir, ann))

print(f"Brazilian: {len(all_tracks['Brazilian'])} tracks")
print(f"Mercosur:  {len(all_tracks['Mercosur'])} tracks")

# Split 90/10 estratificado por layout
train_tracks = []
val_tracks = []

for layout in ['Brazilian', 'Mercosur']:
    tracks = all_tracks[layout][:]
    random.shuffle(tracks)
    n_val = len(tracks) // 10  # 500 por layout
    val_tracks.extend(tracks[:n_val])
    train_tracks.extend(tracks[n_val:])

print(f"\nTreino: {len(train_tracks)} tracks")
print(f"Validacao: {len(val_tracks)} tracks")

# Coletar tracks de teste
test_tracks = []
for track_name in sorted(os.listdir(TEST_DIR)):
    track_dir = os.path.join(TEST_DIR, track_name)
    if not os.path.isdir(track_dir):
        continue
    ann_path = os.path.join(track_dir, 'annotations.json')
    with open(ann_path) as f:
        ann = json.load(f)
    test_tracks.append((track_dir, ann))

print(f"Teste: {len(test_tracks)} tracks")

# Exemplo de anotacao
ex_dir, ex_ann = train_tracks[0]
print(f"\nExemplo: {os.path.basename(ex_dir)}")
print(f"  plate_text: {ex_ann['plate_text']}")
print(f"  plate_layout: {ex_ann['plate_layout']}")
print(f"  imagens com corners: {list(ex_ann['corners'].keys())}")

## 4. Funcoes utilitarias

In [ ]:
import shutil
import cv2
import numpy as np
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Tamanho alvo da retificacao (mesmo recorte do pipeline: 128x64)
WARP_W = 128
WARP_H = 64


def rectify_plate(img_bgr, corners_dict):
    """
    Retifica a placa usando os 4 cantos.
    corners_dict: {'top-left': [x,y], 'top-right': [x,y], ...}
    """
    src = np.array([
        corners_dict['top-left'],
        corners_dict['top-right'],
        corners_dict['bottom-right'],
        corners_dict['bottom-left'],
    ], dtype=np.float32)
    dst = np.array([
        [0, 0], [WARP_W - 1, 0],
        [WARP_W - 1, WARP_H - 1], [0, WARP_H - 1],
    ], dtype=np.float32)
    M = cv2.getPerspectiveTransform(src, dst)
    return cv2.warpPerspective(img_bgr, M, (WARP_W, WARP_H), flags=cv2.INTER_CUBIC)


def prepare_parseq_dataset(tracks, output_dir, split_name, rectify):
    """
    Prepara imagens e labels para um split.
    Se rectify=True, aplica retificacao de perspectiva.
    Se rectify=False, salva a imagem original.
    """
    img_dir = os.path.join(output_dir, 'images', split_name)
    os.makedirs(img_dir, exist_ok=True)
    labels = []
    skipped = 0

    for track_dir, ann in tracks:
        plate_text = ann['plate_text'].strip().upper()
        if not plate_text:
            continue
        corners = ann.get('corners', {})
        track_id = os.path.basename(track_dir)

        for fname in sorted(os.listdir(track_dir)):
            if not fname.lower().endswith(('.png', '.jpg', '.jpeg')):
                continue

            img_path = os.path.join(track_dir, fname)
            img = cv2.imread(img_path)
            if img is None:
                skipped += 1
                continue

            if rectify and fname in corners and corners[fname]:
                img = rectify_plate(img, corners[fname])

            out_name = f"{track_id}_{fname}"
            # Salva como PNG para evitar perda de qualidade
            out_name_png = os.path.splitext(out_name)[0] + '.png'
            cv2.imwrite(os.path.join(img_dir, out_name_png), img)
            labels.append(f"{out_name_png},{plate_text}")

    labels_path = os.path.join(output_dir, f'{split_name}_labels.txt')
    with open(labels_path, 'w') as f:
        f.write('\n'.join(labels))

    print(f"  {split_name}: {len(labels)} imagens, {skipped} puladas")
    return len(labels)


class OCRDataset(Dataset):
    def __init__(self, images_dir, labels_file, transform):
        self.images_dir = images_dir
        self.transform  = transform
        df = pd.read_csv(labels_file, header=None, names=['img_name', 'text'], sep=',')
        df.dropna(subset=['text'], inplace=True)
        df['text'] = df['text'].astype(str)
        self.data = df
        print(f"    {len(self.data)} amostras de {os.path.basename(labels_file)}")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        img_path = os.path.join(self.images_dir, row['img_name'])
        image = Image.open(img_path).convert('RGB')
        return self.transform(image), str(row['text'])


def collate_fn(batch):
    images, texts = zip(*batch)
    return torch.stack(images, 0), texts


def train_parseq(dataset_dir, checkpoint_dir, batch_size=64, lr=1e-5, max_epochs=10):
    """
    Fine-tuna PARSeq Tiny no dataset preparado.
    Retorna o modelo treinado.
    """
    os.makedirs(checkpoint_dir, exist_ok=True)

    print("Carregando PARSeq Tiny pre-treinado...")
    parseq = torch.hub.load('baudm/parseq', 'parseq_tiny', pretrained=True).to(DEVICE)
    parseq.train()
    parseq.hparams.lr = lr

    img_size = parseq.hparams.img_size  # [32, 128]
    transform = T.Compose([
        T.Resize(img_size, T.InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])

    print("Criando DataLoaders...")
    train_ds = OCRDataset(
        os.path.join(dataset_dir, 'images', 'train'),
        os.path.join(dataset_dir, 'train_labels.txt'),
        transform
    )
    val_ds = OCRDataset(
        os.path.join(dataset_dir, 'images', 'val'),
        os.path.join(dataset_dir, 'val_labels.txt'),
        transform
    )

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=2, collate_fn=collate_fn)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                              num_workers=2, collate_fn=collate_fn)

    checkpoint_cb = pl.callbacks.ModelCheckpoint(
        dirpath=checkpoint_dir,
        filename='parseq-{epoch:02d}-{val_loss:.4f}',
        save_top_k=3,
        monitor='val_loss',
        mode='min',
        save_last=True,
    )

    trainer = pl.Trainer(
        max_epochs=max_epochs,
        accelerator=DEVICE,
        log_every_n_steps=10,
        callbacks=[
            pl.callbacks.EarlyStopping(monitor='val_loss', patience=3, mode='min'),
            checkpoint_cb,
        ]
    )

    last_ckpt = os.path.join(checkpoint_dir, 'last.ckpt')
    if os.path.exists(last_ckpt):
        print(f"Retomando de: {last_ckpt}")
        trainer.fit(parseq, train_loader, val_loader, ckpt_path=last_ckpt)
    else:
        print("Iniciando fine-tuning do zero.")
        trainer.fit(parseq, train_loader, val_loader)

    print("Fine-tuning concluido!")
    return parseq


from collections import Counter, defaultdict


def mvcp(predictions):
    """
    predictions: lista de (texto, tensor_de_scores_por_char)
    Retorna a predicao final da track via Majority Vote by Character Position.
    (Mesma funcao dos notebooks de PARSeq individual.)
    """
    if not predictions:
        return ""

    # remove caracteres nao alfanumericos
    clean = []
    for text, scores in predictions:
        text_clean = []
        scores_clean = []
        for char, score in zip(text, scores):
            if char.isalnum():
                text_clean.append(char)
                scores_clean.append(score)
        clean.append(("".join(text_clean), scores_clean))

    # comprimento mais comum entre as predicoes
    length = Counter(len(t) for t, _ in clean).most_common(1)[0][0]

    # filtra predicoes com esse comprimento
    clean = [(t, s) for t, s in clean if len(t) == length]

    final_chars = []
    for pos in range(length):
        # coleta (caractere, confianca) de cada frame nessa posicao
        char_scores = []
        for text, scores in clean:
            char = text[pos]
            score = scores[pos].item() if pos < len(scores) else 0.0
            char_scores.append((char, score))

        vote_count = Counter(c for c, _ in char_scores)
        max_votes = max(vote_count.values())
        candidates = [c for c, v in vote_count.items() if v == max_votes]

        if len(candidates) == 1:
            final_chars.append(candidates[0])
        else:
            # desempate pela media de confianca
            best_char = max(
                candidates,
                key=lambda c: sum(s for ch, s in char_scores if ch == c) / vote_count[c]
            )
            final_chars.append(best_char)

    return "".join(final_chars)


def evaluate_parseq(model, test_base):
    """
    Avalia o modelo no teste oficial, por track (MVCP sobre os 5 quadros),
    como nos notebooks de PARSeq individual.
    Retorna lista de resultados (um por track) e a taxa de reconhecimento global.
    """
    model.to(DEVICE).eval()
    img_size = model.hparams.img_size
    transform = T.Compose([
        T.Resize(img_size, T.InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])

    results = []
    for track_name in sorted(os.listdir(test_base)):
        track_path = os.path.join(test_base, track_name)
        if not os.path.isdir(track_path):
            continue

        annotation_path = os.path.join(track_path, 'annotations.json')
        if not os.path.exists(annotation_path):
            print(f"  aviso: sem anotacao em {track_name}, pulando.")
            continue

        with open(annotation_path, 'r') as f:
            ann = json.load(f)
        gt_plate = ann.get('plate_text', '').upper()
        category = ann.get('plate_layout', 'unknown')
        if not gt_plate:
            continue

        frames = sorted([f for f in os.listdir(track_path)
                         if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        if not frames:
            continue

        # acumula predicoes e scores de todos os frames da track
        track_predictions = []
        for img_name in frames:
            image = cv2.imread(os.path.join(track_path, img_name))
            if image is None:
                continue
            plate_pil = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
            plate_tensor = transform(plate_pil).unsqueeze(0).to(DEVICE)

            with torch.no_grad():
                logits = model(images=plate_tensor)

            texts, scores = model.tokenizer.decode(logits.cpu())
            track_predictions.append((texts[0], scores[0]))

        results.append({
            'track': track_name,
            'layout': category,
            'gt': gt_plate,
            'pred': mvcp(track_predictions),
        })

    correct = sum(1 for r in results if r['gt'] == r['pred'])
    accuracy = correct / len(results) * 100 if results else 0.0
    return results, accuracy


def rate_by_layout(results):
    """Taxa de reconhecimento por track, separada por leiaute."""
    total = defaultdict(int)
    correct = defaultdict(int)
    for r in results:
        total[r['layout']] += 1
        if r['gt'] == r['pred']:
            correct[r['layout']] += 1
    return {cat: (correct[cat] / total[cat] * 100, correct[cat], total[cat])
            for cat in sorted(total)}


print("Funcoes utilitarias definidas.")
print(f"Device: {DEVICE}")
print(f"Tamanho de retificacao: {WARP_W}x{WARP_H}")

## 5. Comparacao visual: retificado vs original

Alguns exemplos lado a lado para verificar o efeito da retificacao.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(4, 2, figsize=(10, 8))
fig.suptitle('Original (esquerda) vs Retificado (direita)', fontsize=14)

for i, (track_dir, ann) in enumerate(train_tracks[:4]):
    corners = ann.get('corners', {})
    # Pega a primeira imagem com corners
    for fname in sorted(os.listdir(track_dir)):
        if fname in corners and corners[fname] and fname.endswith(('.png', '.jpg')):
            img_path = os.path.join(track_dir, fname)
            img = cv2.imread(img_path)
            if img is not None:
                img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                rect_rgb = cv2.cvtColor(rectify_plate(img, corners[fname]), cv2.COLOR_BGR2RGB)

                axes[i, 0].imshow(img_rgb)
                axes[i, 0].set_title(f"{ann['plate_text']} — {img.shape[1]}x{img.shape[0]}")
                axes[i, 0].axis('off')

                axes[i, 1].imshow(rect_rgb)
                axes[i, 1].set_title(f"Retificado — {WARP_W}x{WARP_H}")
                axes[i, 1].axis('off')
                break

plt.tight_layout()
plt.show()

---
## 6. Experimento 1: Treino COM retificacao

In [ ]:
RECT_DIR = '/content/parseq_rect'

if os.path.exists(RECT_DIR):
    shutil.rmtree(RECT_DIR)

print("Preparando dataset RETIFICADO...")
prepare_parseq_dataset(train_tracks, RECT_DIR, 'train', rectify=True)
prepare_parseq_dataset(val_tracks,   RECT_DIR, 'val',   rectify=True)
print("Dataset retificado pronto.")

In [ ]:
# ── Ajuste o caminho de checkpoints conforme sua configuracao ──
CKPT_RECT = '/content/drive/MyDrive/<CAMINHO>/checkpoints_retificado_10ep'

model_rect = train_parseq(
    dataset_dir=RECT_DIR,
    checkpoint_dir=CKPT_RECT,
    batch_size=64,
    lr=1e-5,
    max_epochs=10,
)

---
## 7. Experimento 2: Treino SEM retificacao

In [ ]:
RAW_DIR = '/content/parseq_raw'

if os.path.exists(RAW_DIR):
    shutil.rmtree(RAW_DIR)

print("Preparando dataset SEM RETIFICACAO...")
prepare_parseq_dataset(train_tracks, RAW_DIR, 'train', rectify=False)
prepare_parseq_dataset(val_tracks,   RAW_DIR, 'val',   rectify=False)
print("Dataset sem retificacao pronto.")

In [ ]:
# ── Ajuste o caminho de checkpoints conforme sua configuracao ──
CKPT_RAW = '/content/drive/MyDrive/<CAMINHO>/checkpoints_sem_retificacao_10ep'

model_raw = train_parseq(
    dataset_dir=RAW_DIR,
    checkpoint_dir=CKPT_RAW,
    batch_size=64,
    lr=1e-5,
    max_epochs=10,
)

---
## 8. Preparar conjunto de teste e avaliar ambos os modelos

O teste usa as imagens originais (sem retificacao, pois nao ha corners).
Ambos os modelos sao avaliados no mesmo conjunto, por track (MVCP),
lendo direto das pastas do teste oficial.

In [ ]:
print("--- Avaliando modelo COM retificacao (por track, MVCP) ---")
results_rect, acc_rect = evaluate_parseq(model_rect, TEST_DIR)
print(f"Recognition Rate: {acc_rect:.2f}%  ({sum(1 for r in results_rect if r['gt']==r['pred'])}/{len(results_rect)} tracks)")

print("\n--- Avaliando modelo SEM retificacao (por track, MVCP) ---")
results_raw, acc_raw = evaluate_parseq(model_raw, TEST_DIR)
print(f"Recognition Rate: {acc_raw:.2f}%  ({sum(1 for r in results_raw if r['gt']==r['pred'])}/{len(results_raw)} tracks)")

layout_rect = rate_by_layout(results_rect)
layout_raw  = rate_by_layout(results_raw)

print("\n" + "="*65)
print(f"{'Modelo':<25} {'Global':>10} " + " ".join(f"{cat:>12}" for cat in layout_rect))
print("-"*65)
print(f"{'COM retificacao':<25} {acc_rect:>9.2f}% " + " ".join(f"{layout_rect[c][0]:>11.2f}%" for c in layout_rect))
print(f"{'SEM retificacao':<25} {acc_raw:>9.2f}% " + " ".join(f"{layout_raw[c][0]:>11.2f}%" for c in layout_raw))
print(f"{'Diferenca':<25} {acc_rect - acc_raw:>+9.2f}pp " + " ".join(f"{layout_rect[c][0]-layout_raw[c][0]:>+10.2f}pp" for c in layout_rect))
print("="*65)


## 9. (Opcional) Carregar modelos salvos

Execute esta celula para pular o treino e carregar modelos ja salvos.

In [ ]:
# Descomente para carregar modelos ja treinados:

# SAVE_BASE = '/content/drive/MyDrive/<CAMINHO>'
#
# print("Carregando modelo COM retificacao...")
# model_rect = torch.hub.load('baudm/parseq', 'parseq_tiny', pretrained=False).to(DEVICE)
# model_rect.model.load_state_dict(torch.load(
#     os.path.join(SAVE_BASE, 'parseq_tiny_lrlpr26_retificado_10ep.pt'),
#     map_location=DEVICE
# ))
#
# print("Carregando modelo SEM retificacao...")
# model_raw = torch.hub.load('baudm/parseq', 'parseq_tiny', pretrained=False).to(DEVICE)
# model_raw.model.load_state_dict(torch.load(
#     os.path.join(SAVE_BASE, 'parseq_tiny_lrlpr26_sem_retificacao_10ep.pt'),
#     map_location=DEVICE
# ))
#
# print("Modelos carregados.")

---
## 10. Analise detalhada

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from collections import defaultdict

# ── CER ────────────────────────────────────────────────────────────────────
def cer(gt, pred):
    """CER via distancia de Levenshtein normalizada."""
    n, m = len(gt), len(pred)
    dp = list(range(m + 1))
    for i in range(1, n + 1):
        prev = dp[:]
        dp[0] = i
        for j in range(1, m + 1):
            cost = 0 if gt[i-1] == pred[j-1] else 1
            dp[j] = min(dp[j] + 1, dp[j-1] + 1, prev[j-1] + cost)
    return dp[m] / max(n, 1)


def detailed_analysis(results, model_name):
    """Imprime metricas detalhadas e gera graficos."""
    cer_vals = [cer(r['gt'], r['pred']) for r in results]
    mean_cer = np.mean(cer_vals) * 100
    correct = sum(1 for r in results if r['gt'] == r['pred'])
    accuracy = correct / len(results) * 100

    print(f"\n{'='*50}")
    print(f"  {model_name}")
    print(f"{'='*50}")
    print(f"  Recognition Rate (por track): {accuracy:.2f}% ({correct}/{len(results)})")
    for cat, (rr, c, t) in rate_by_layout(results).items():
        print(f"    {cat}: {rr:.2f}% ({c}/{t})")
    print(f"  CER medio: {mean_cer:.2f}%")
    print(f"  CER=0:     {sum(v == 0 for v in cer_vals)}")
    print(f"  CER>50%:   {sum(v > 0.5 for v in cer_vals)}")

    # Erros por quantidade de caracteres errados
    n_wrong_chars = defaultdict(int)
    for r in results:
        gt, pred = r['gt'], r['pred']
        wrong = sum(1 for g, p in zip(gt, pred) if g != p)
        wrong += abs(len(gt) - len(pred))
        if wrong > 0:
            n_wrong_chars[wrong] += 1

    if n_wrong_chars:
        print(f"\n  Distribuicao de erros:")
        for n_err in sorted(n_wrong_chars.keys()):
            print(f"    {n_err} char(s) errado(s): {n_wrong_chars[n_err]} tracks")

    # Top confusoes
    CHARS = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789'
    char2idx = {c: i for i, c in enumerate(CHARS)}
    confusion = np.zeros((len(CHARS), len(CHARS)), dtype=int)
    for r in results:
        for g, p in zip(r['gt'], r['pred']):
            if g in char2idx and p in char2idx and g != p:
                confusion[char2idx[g], char2idx[p]] += 1

    erros = [(confusion[i, j], CHARS[i], CHARS[j])
             for i in range(len(CHARS)) for j in range(len(CHARS))
             if i != j and confusion[i, j] > 0]
    if erros:
        print(f"\n  Top 10 confusoes:")
        for cnt, real, pred in sorted(erros, reverse=True)[:10]:
            print(f"    {real} -> {pred}: {cnt}x")

    # Acuracia por posicao
    max_len = max((len(r['gt']) for r in results), default=0)
    pos_correct = defaultdict(int)
    pos_total   = defaultdict(int)
    for r in results:
        for i, g in enumerate(r['gt']):
            pos_total[i] += 1
            if i < len(r['pred']) and r['pred'][i] == g:
                pos_correct[i] += 1

    return {
        'accuracy': accuracy,
        'cer': mean_cer,
        'cer_vals': cer_vals,
        'confusion': confusion,
        'pos_correct': pos_correct,
        'pos_total': pos_total,
    }


stats_rect = detailed_analysis(results_rect, 'COM RETIFICACAO')
stats_raw  = detailed_analysis(results_raw,  'SEM RETIFICACAO')

In [ ]:
# ── Graficos comparativos ─────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1) Barras: Accuracy e CER
ax = axes[0]
x = [0, 1]
acc_vals = [stats_rect['accuracy'], stats_raw['accuracy']]
bars = ax.bar(x, acc_vals, color=['#2196F3', '#FF9800'], width=0.5)
ax.set_xticks(x)
ax.set_xticklabels(['Com\nretificacao', 'Sem\nretificacao'])
ax.set_ylabel('Accuracy (%)')
ax.set_title('Accuracy no Teste')
ax.set_ylim(0, 105)
for bar, val in zip(bars, acc_vals):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.5,
            f'{val:.2f}%', ha='center', va='bottom', fontweight='bold')

# 2) Histograma de CER sobrepostos
ax = axes[1]
ax.hist([v*100 for v in stats_rect['cer_vals']], bins=20, alpha=0.6,
        color='#2196F3', label=f"Retificado (CER={stats_rect['cer']:.2f}%)")
ax.hist([v*100 for v in stats_raw['cer_vals']], bins=20, alpha=0.6,
        color='#FF9800', label=f"Original (CER={stats_raw['cer']:.2f}%)")
ax.set_xlabel('CER (%)')
ax.set_ylabel('Tracks')
ax.set_title('Distribuicao do CER')
ax.legend()

# 3) Acuracia por posicao
ax = axes[2]
positions = sorted(stats_rect['pos_total'].keys())
pos_acc_rect = [stats_rect['pos_correct'][p] / stats_rect['pos_total'][p] * 100 for p in positions]
pos_acc_raw  = [stats_raw['pos_correct'][p]  / stats_raw['pos_total'][p]  * 100 for p in positions]

w = 0.35
x_pos = np.arange(len(positions))
ax.bar(x_pos - w/2, pos_acc_rect, w, color='#2196F3', label='Retificado')
ax.bar(x_pos + w/2, pos_acc_raw,  w, color='#FF9800', label='Original')
ax.set_xticks(x_pos)
ax.set_xticklabels([str(p+1) for p in positions])
ax.set_xlabel('Posicao do caractere')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Accuracy por Posicao')
ax.set_ylim(0, 105)
ax.legend()

plt.tight_layout()
plt.show()

# ── Tabela resumo final ──────────────────────────────────────────────────
print("\n" + "="*60)
print(f"{'RESUMO FINAL':^60}")
print("="*60)
print(f"{'Metrica':<30} {'Retificado':>12} {'Original':>12}")
print("-"*60)
print(f"{'Accuracy (%)':<30} {stats_rect['accuracy']:>11.2f}% {stats_raw['accuracy']:>11.2f}%")
print(f"{'CER medio (%)':<30} {stats_rect['cer']:>11.2f}% {stats_raw['cer']:>11.2f}%")
print(f"{'Diferenca Accuracy (pp)':<30} {stats_rect['accuracy'] - stats_raw['accuracy']:>+11.2f}")
print(f"{'Diferenca CER (pp)':<30} {stats_rect['cer'] - stats_raw['cer']:>+11.2f}")
print("="*60)

---
## 11. Salvar modelos no Drive

In [ ]:
# ── Ajuste o caminho de saida conforme sua configuracao ──
SAVE_BASE = '/content/drive/MyDrive/<CAMINHO>'
os.makedirs(SAVE_BASE, exist_ok=True)

path_rect = os.path.join(SAVE_BASE, 'parseq_tiny_lrlpr26_retificado_10ep.pt')
torch.save(model_rect.model.state_dict(), path_rect)
print(f"Salvo: {path_rect}")

path_raw = os.path.join(SAVE_BASE, 'parseq_tiny_lrlpr26_sem_retificacao_10ep.pt')
torch.save(model_raw.model.state_dict(), path_raw)
print(f"Salvo: {path_raw}")

import json as json_mod

report = {
    'experimento': 'Impacto da retificacao no OCR — LRLPR-26 Cenario A',
    'dataset': 'LRLPR-26 Scenario-A',
    'train_tracks': len(train_tracks),
    'val_tracks': len(val_tracks),
    'test_tracks': len(test_tracks),
    'modelo': 'PARSeq Tiny',
    'hiperparametros': {'max_epochs': 10, 'patience': 3, 'lr': 1e-5, 'batch_size': 64},
    'avaliacao': 'por track (MVCP sobre os quadros), teste oficial',
    'resultados': {
        'com_retificacao': {
            'accuracy': round(stats_rect['accuracy'], 2),
            'cer': round(stats_rect['cer'], 2),
            'por_layout': {c: round(v[0], 2) for c, v in rate_by_layout(results_rect).items()},
        },
        'sem_retificacao': {
            'accuracy': round(stats_raw['accuracy'], 2),
            'cer': round(stats_raw['cer'], 2),
            'por_layout': {c: round(v[0], 2) for c, v in rate_by_layout(results_raw).items()},
        },
        'diferenca_accuracy_pp': round(stats_rect['accuracy'] - stats_raw['accuracy'], 2),
        'diferenca_cer_pp': round(stats_rect['cer'] - stats_raw['cer'], 2),
    }
}

report_path = os.path.join(SAVE_BASE, 'comparacao_retificacao_10ep.json')
with open(report_path, 'w') as f:
    json_mod.dump(report, f, indent=2, ensure_ascii=False)
print(f"Relatorio: {report_path}")